# Milestone 2: convolutional image tokens in the recurrent transformer

Replace direct patch projection with five convolutional layers and regional pooling.
Keep **16 image tokens, R=2, questions, transformer and loss unchanged**.
**8,640 updates / 40 presentations per QA / 276,480 QA presentations**.

290,752 parameters. Matched exposure does not mean matched compute.
Both source archives must be pushed to the repo root. Enable GPU and internet;
set a committed `REPO_REF`. No reserved validation/test inference.
See `docs/milestones/milestone2_conv_stem.md` for the fixed protocol.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # Commit SHA preferred; must contain this implementation.
REPO_DIR = Path("/kaggle/working/multi-modal-loop-conv-stem")
RUN_ROOT = Path("/kaggle/working/milestone2_conv_stem")
TRANSFORMER_SOURCE = REPO_DIR / "milestone2_multi_arrangement_artifacts.zip"
CNN_SOURCE = REPO_DIR / "milestone2_cnn_baseline_artifacts.zip"

## Checkout and install

Run cells in order. Keep Kaggle’s installed PyTorch.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit references and prepare

Audit both saved references without loading their weights. Preserve the exact
four training and four transfer arrangements and complete size quartets.
The model's non-image initialization matches a fresh baseline under seed 0;
the new stem uses its own preserved seed-0 initialization.


In [ ]:
import os

import torch

from multimodal_loop.eval.kaggle_conv_stem import (
    archive_conv_stem,
    prepare_conv_stem,
    run_conv_stem,
)

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
torch.set_num_threads(2)
for source in (TRANSFORMER_SOURCE, CNN_SOURCE):
    if not source.is_file():
        raise FileNotFoundError(f"Push the completed archive to the repo root: {source}")
run = prepare_conv_stem(REPO_DIR, RUN_ROOT, TRANSFORMER_SOURCE, CNN_SOURCE)
print("Manifest:", run.manifest_sha256)
print("Budget: 8,640 updates; 40 presentations per QA. Training fit monitored only.")

## Train and assess the final checkpoint

One CUDA device, float32. No resume, early stopping, budget extension, or
best-checkpoint selection. Train all components from scratch; no reference weights.


In [ ]:
report = run_conv_stem(run)

## Read the results

Check training fit first, then every transfer arrangement and shape. A gain
concentrated in one row is not broad transfer. Correct families require both
circle/square answers correct across all four sizes. Transfer has no new pass/fail gate.


In [ ]:
import json

from IPython.display import Markdown, display

comparison = json.loads((RUN_ROOT / "diagnosis/comparison.json").read_text())
print("Training criteria:", "PASS" if report["assessment"]["passed"] else "NOT MET")
for role in ("training_fit", "transfer"):
    m = report["aggregates"][role]
    print(
        f"{role}: {m['summary']['accuracy']:.2%} accuracy; "
        f"{m['families']['correct']}/{m['families']['total']} correct families"
    )
rows = [
    "| Arrangement | Patches | CNN | Conv stem | Circle | Square | Triangle | Families |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for name, result in comparison["transformer"]["arrangements"].items():
    m = report["arrangements"][name]
    shapes = {r["shape"]: r["accuracy"] for r in m["summary"]["breakdowns"]["shape"]}
    cnn = comparison["cnn"]["arrangements"][name]["accuracy"]["reference"]
    rows.append(
        f"| {name} | {result['accuracy']['reference']:.2%} | {cnn:.2%} | "
        f"{m['summary']['accuracy']:.2%} | {shapes['circle']:.2%} | "
        f"{shapes['square']:.2%} | {shapes['triangle']:.2%} | "
        f"{m['families']['correct']}/{m['families']['total']} |"
    )
display(Markdown("\n".join(rows)))
for directory in ("training", "diagnosis"):
    print(directory, json.loads((RUN_ROOT / directory / "runtime.json").read_text()))
print("Selected errors:", RUN_ROOT / "diagnosis/inspection.html")
print("Equal exposure, unequal compute. No recurrence or milestone-completion claim.")

## Download artifacts

Return this archive to the local repository root for review.


In [ ]:
from IPython.display import FileLink, display

archive = archive_conv_stem(run)
print(archive)
display(FileLink(str(archive)))